# VisiShield-Edge: NFL Big Data Bowl 2027

**10 Hz movement features, cut efficiency, and rookie-season performance.**

This notebook is a reproducible report scaffold for the project's modular pipeline. It does not download competition data, invent results, or claim findings from mock tests. Run it where the repository modules and authorized data are available.

## 1. Environment and bounded local inputs

Locate the project modules and list expected local inputs. No raw data are read in this setup cell. Keep tracking input at or below the pipeline's 250,000-row default limit, and use complete player tracks rather than truncating a larger file.

In [ ]:
from pathlib import Path
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'src' / 'pipeline.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the repository root containing src/pipeline.py. '
        'Open this notebook from the project checkout or configure PROJECT_ROOT.'
    )

SRC = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
TRACKING_CSV = DATA_DIR / 'combine_tracking.csv'
PERFORMANCE_CSV = DATA_DIR / 'player_season_outcomes.csv'
sys.path.insert(0, str(SRC))

print(f'Project root: {PROJECT_ROOT}')
print(f'Tracking input present: {TRACKING_CSV.is_file()}')
print(f'Performance input present: {PERFORMANCE_CSV.is_file()}')
print('No input data have been loaded.')

## 2. Pipeline self-validation

Run the pipeline's synthetic-data tests. These exercise validation and feature/model workflow behavior without requiring competition files.

In [ ]:
pipeline_test = subprocess.run(
    [sys.executable, str(SRC / 'pipeline.py'), '--self-test'],
    cwd=PROJECT_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(pipeline_test.stdout)
if pipeline_test.stderr:
    print(pipeline_test.stderr)
if pipeline_test.returncode != 0:
    raise RuntimeError('Pipeline self-test failed.')
print('Pipeline self-test passed.')

## 3. Optional local pipeline run

When the bounded tracking CSV exists, run feature extraction and, when supplied, the pre-aggregated player-season outcome model. The pipeline validates required columns and rejects inputs over its configured row limit rather than silently truncating tracks. Rookie filtering uses `draft_year` and `season == draft_year`; check the source schema and cohort before interpreting results.

In [ ]:
if TRACKING_CSV.is_file():
    from pipeline import run_full_pipeline

    performance_input = PERFORMANCE_CSV if PERFORMANCE_CSV.is_file() else None
    result = run_full_pipeline(
        tracking_data=TRACKING_CSV,
        performance_data=performance_input,
    )
    print(f'Player summaries: {len(result.player_summary):,} rows')
    if result.model_metrics is None:
        print('No performance outcomes supplied; model evaluation was skipped.')
    else:
        display(result.model_metrics)
else:
    print('Skipped: add an authorized, bounded tracking CSV under data/.')

## 4. Visualization validation and available artifacts

The visualization self-test uses mock inputs and removes temporary test artifacts. Production figures are written only to this repository's `outputs/` directory. Actual-vs-predicted plots require a separate paired out-of-sample prediction table; aggregate cross-validation metrics alone are not sufficient.

In [ ]:
visualization_test = subprocess.run(
    [sys.executable, str(SRC / 'visualization.py'), '--self-test'],
    cwd=PROJECT_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(visualization_test.stdout)
if visualization_test.stderr:
    print(visualization_test.stderr)
if visualization_test.returncode != 0:
    raise RuntimeError('Visualization self-test failed.')

plot_tracking = OUTPUTS_DIR / 'tracking_features.csv'
plot_predictions = OUTPUTS_DIR / 'model_predictions.csv'
if plot_tracking.is_file() or plot_predictions.is_file():
    from visualization import generate_project_visualizations

    plots = generate_project_visualizations(
        tracking_data=plot_tracking if plot_tracking.is_file() else None,
        predictions_data=plot_predictions if plot_predictions.is_file() else None,
    )
    for plot_name, plot_path in plots.items():
        print(f'{plot_name}: {plot_path}')
else:
    print('No prepared visualization CSVs found in outputs/; no figures generated.')
print('Visualization self-test passed.')

## 5. Interpretation and limitations

- Treat correlations and cross-validation scores as descriptive evidence, not causal claims.
- Confirm player identifiers, measurement units, table grain, missing-value conventions, and the rookie-season join against the authorized competition schema.
- Do not interpret synthetic self-test output as competition results.
- The current model pipeline saves aggregate evaluation metrics; it does not persist per-player predictions. Build and review an out-of-fold prediction table before using actual-vs-predicted charts.
- Keep raw and generated data local, and have a human reviewer approve cohort decisions and strategic conclusions.